# Capstone Project 2 — Financial Forecasting Frontier
## Part 1: Data Analysis & Management (Hadoop + Hive)

**Objective:** Use Hadoop and Hive to store and query large volumes of banking data efficiently, simulating how banks manage their vast data repositories.

**Dataset:** `bank.csv` — 4,521 customer records, 17 features (UCI Bank Marketing dataset)

This notebook documents the HDFS storage setup and Hive querying layer of the pipeline. The actual HDFS/Hive commands are run from the shell (`hdfs/hdfs_setup.sh`, `hive/bank_hive.sql`); this notebook captures the commands, their purpose, and the resulting output for submission/review purposes.

### 1.1 Why Hadoop + Hive for this dataset?

Although `bank.csv` is only ~360 KB, the project is designed to demonstrate the **same architecture banks use for multi-terabyte transaction data**. The pipeline:

- Stores raw data on **HDFS** (Hadoop Distributed File System) — fault-tolerant, replicated 3×, splits files into 128 MB blocks
- Queries it with **Hive SQL** — a SQL layer on top of HDFS, compiled into distributed jobs
- This separates **storage** (HDFS) from **querying** (Hive) from **processing/ML** (Spark) — the standard layered big-data architecture

In [ ]:
# Cell: View the HDFS setup script
with open("../hdfs/hdfs_setup.sh") as f:
    print(f.read())

### 1.2 HDFS Directory Structure Created

```
/user/bankproject/
├── raw/bank/bank.csv          ← Original CSV (uploaded via hdfs dfs -put)
├── processed/features/        ← Parquet after EDA + feature engineering
├── processed/encoded/         ← Encoded features for ML
├── models/                    ← Saved Spark ML models (LR, DT, RF)
├── hive/warehouse/            ← Hive managed/external tables
├── streaming/input|output/    ← Real-time streaming records
└── reports/                   ← model_results.json and summaries
```

**Commands executed (see `hdfs_setup.sh`):**
```bash
start-dfs.sh && start-yarn.sh
hdfs dfs -mkdir -p /user/bankproject/raw/bank
hdfs dfs -put -f data/bank.csv /user/bankproject/raw/bank/bank.csv
hdfs dfs -ls /user/bankproject/raw/bank/
hdfs dfs -du -h /user/bankproject/raw/bank/
hdfs dfs -chmod -R 755 /user/bankproject
```

In [ ]:
# Cell: View the Hive DDL + analytics SQL
with open("../hive/bank_hive.sql") as f:
    print(f.read())

### 1.3 Hive Schema Design Decisions

- **`EXTERNAL TABLE`** for `bank_raw` — Hive only stores metadata; dropping the table does NOT delete the underlying HDFS file. Critical for production data shared across tools.
- **ORC + Snappy** for `bank_orc` — columnar storage gives 60–70% compression and predicate pushdown (a query reading 3 of 17 columns processes ~80% less data than the equivalent CSV scan).
- **Column rename**: `default` → `def` because `DEFAULT` is a reserved keyword in HiveQL (same issue handled in PySpark via `withColumnRenamed`).

### 1.4 Key Query Results (executed via `hive -f hive/bank_hive.sql`)

**Subscription rate by job (Q1):**

| Job | Sub Rate |
|-----|----------|
| retired | 23.5% |
| student | 22.6% |
| management | 13.5% |
| blue-collar | 7.3% |

**Subscription rate by month (Q6):**

| Month | Rate |
|-------|------|
| October | 46.3% |
| December | 45.0% |
| March | 42.9% |
| May | 6.7% |

These results confirm that **timing and customer segment** materially affect campaign success — directly useful for the EDA and feature engineering steps in the next notebook.

### 1.5 Summary

| Task | Tool | Status |
|------|------|--------|
| Raw data ingestion to HDFS | `hdfs dfs -put` | ✅ Done |
| Directory structure for raw/processed/models/streaming | HDFS | ✅ Done |
| External table over raw CSV | Hive | ✅ Done |
| Optimized ORC table (Snappy compression) | Hive | ✅ Done |
| Segmentation queries (job, education, age-group) | HiveQL | ✅ Done |
| Campaign performance queries (month, duration, poutcome) | HiveQL | ✅ Done |

Next: **`02_eda_feature_engineering_spark.ipynb`** — exploratory data analysis and feature engineering using PySpark.